In [2]:
import polars as pl
import urllib.request
import json
import os
from pathlib import Path

# Config polars to not truncate the columns

In [4]:
pl.Config(tbl_width_chars=200)
pl.Config(tbl_cols=-1) 

In [3]:
print(os.getcwd())

/home/jovyan/Portfolio/CBS_elekprod_vs_emissies


# Setup variables to download data

In [5]:
directory = Path.cwd() / "datafiles"
directory.mkdir(parents=True, exist_ok=True)

urls = [
    { 
        "name": "elektriciteitsprod_naar_energiedrager_dataset",
        "url": "https://opendata.cbs.nl/ODataFeed/odata/86266NED/UntypedDataSet?$format=json",
        "types": {"ID": pl.UInt16, "CentraleDecentraleProductie": pl.String  ,"Energiedragers": pl.String, "Perioden": pl.String  ,"TotaalElektriciteitEnWarmteTJ_1": pl.UInt32  ,"ElektriciteitGWh_2": pl.UInt32  ,"ElektriciteitTJ_3": pl.UInt32  ,"Elektriciteit_4": pl.Float64  ,"WarmteTJ_5": pl.UInt32  ,"InzetTJ_6": pl.UInt32,"ElektrischVermogenMW_7": pl.UInt32},
    },{ 
        "name": "elektriciteitsprod_naar_energiedrager_properties",
        "url": "https://opendata.cbs.nl/ODataFeed/odata/86266NED/DataProperties?$format=json",
    },{ 
        "name": "elektriciteitsprod_naar_energiedrager_categorygroups",
        "url": "https://opendata.cbs.nl/ODataFeed/odata/86266NED/CategoryGroups?$format=json",
    },{ 
        "name": "elektriciteitsprod_naar_energiedrager_centdecent",
        "url": "https://opendata.cbs.nl/ODataFeed/odata/86266NED/CentraleDecentraleProductie?$format=json",
    },{ 
        "name": "elektriciteitsprod_naar_energiedrager_dragers",
        "url": "https://opendata.cbs.nl/ODataFeed/odata/86266NED/Energiedragers?$format=json",
    },{ 
        "name": "elektriciteitsprod_naar_energiedrager_perioden",
        "url": "https://opendata.cbs.nl/ODataFeed/odata/86266NED/Perioden?$format=json",
    },{
        "name": "emissies_naar_lucht_dataset",
        "url": "https://opendata.cbs.nl/ODataFeed/odata/85668NED/UntypedDataSet?$filter=((Emissiebronnen+eq+%27T001176++%27)+or+(Emissiebronnen+eq+%27A025447++%27)+or+(Emissiebronnen+eq+%27800044+++%27)+or+(Emissiebronnen+eq+%27305800+++%27)+or+(Emissiebronnen+eq+%27320300+++%27)+or+(Emissiebronnen+eq+%27346700+++%27)+or+(Emissiebronnen+eq+%27800045+++%27)+or+(Emissiebronnen+eq+%27307610+++%27)+or+(Emissiebronnen+eq+%27312505+++%27)+or+(Emissiebronnen+eq+%27315800+++%27)+or+(Emissiebronnen+eq+%27317100+++%27)+or+(Emissiebronnen+eq+%27318600+++%27)+or+(Emissiebronnen+eq+%27320800+++%27)+or+(Emissiebronnen+eq+%27800046+++%27)+or+(Emissiebronnen+eq+%27324600+++%27)+or+(Emissiebronnen+eq+%27328100+++%27)+or+(Emissiebronnen+eq+%27800049+++%27)+or+(Emissiebronnen+eq+%27339205+++%27)+or+(Emissiebronnen+eq+%27800050+++%27)+or+(Emissiebronnen+eq+%27350000+++%27)+or+(Emissiebronnen+eq+%27800051+++%27)+or+(Emissiebronnen+eq+%27301100+++%27)+or+(Emissiebronnen+eq+%27348000+++%27)+or+(Emissiebronnen+eq+%27300008+++%27))+and+((EmissiesNaarLucht+eq+%27A044109%27)+or+(EmissiesNaarLucht+eq+%27A044112%27)+or+(EmissiesNaarLucht+eq+%27A044114%27)+or+(EmissiesNaarLucht+eq+%27A044113%27)+or+(EmissiesNaarLucht+eq+%27A051771%27)+or+(EmissiesNaarLucht+eq+%27A044108%27))&$format=json",
    },{
        "name": "emissies_naar_lucht_bronnen",
        "url": "https://opendata.cbs.nl/ODataFeed/odata/85668NED/Emissiebronnen?$format=json",
    },{
        "name": "emissies_naar_lucht_stoffen",
        "url": "https://opendata.cbs.nl/ODataFeed/odata/85668NED/EmissiesNaarLucht?$format=json",
    },{
        "name": "emissies_naar_lucht_perioden",
        "url": "https://opendata.cbs.nl/ODataFeed/odata/85668NED/Perioden?$format=json",
    },
    ]

# Download necessary data and save it to files to reduce api calls

In [6]:
for dataset in urls:
    with urllib.request.urlopen(dataset["url"]) as response:
        raw_data = response.read()
        json_data = json.loads(raw_data)
        df = pl.DataFrame(json_data["value"])
        df = df.with_columns(
            pl.col(pl.String).str.strip_chars().replace(["", "."], None))
        df.write_csv(directory / f"{dataset['name']}.csv", separator=";")

In [7]:
df = pl.read_csv(f"{directory}/elektriciteitsprod_naar_energiedrager_dataset.csv", separator=";")
print(df.head())

shape: (5, 11)
┌─────┬───────────────────────────┬────────────────┬──────────┬───────────────────────────┬────────────────────┬───────────────────┬─────────────────┬────────────┬───────────┬────────────────────────┐
│ ID  ┆ CentraleDecentraleProduct ┆ Energiedragers ┆ Perioden ┆ TotaalElektriciteitEnWarm ┆ ElektriciteitGWh_2 ┆ ElektriciteitTJ_3 ┆ Elektriciteit_4 ┆ WarmteTJ_5 ┆ InzetTJ_6 ┆ ElektrischVermogenMW_7 │
│ --- ┆ ie                        ┆ ---            ┆ ---      ┆ teTJ_…                    ┆ ---                ┆ ---               ┆ ---             ┆ ---        ┆ ---       ┆ ---                    │
│ i64 ┆ ---                       ┆ str            ┆ str      ┆ ---                       ┆ i64                ┆ i64               ┆ f64             ┆ i64        ┆ i64       ┆ i64                    │
│     ┆ str                       ┆                ┆          ┆ i64                       ┆                    ┆                   ┆                 ┆            ┆           ┆      

In [8]:
dataset = pl.scan_csv(
    directory / "elektriciteitsprod_naar_energiedrager_dataset.csv",
    separator=";",
    schema_overrides=urls[0]["types"],
)

perioden = pl.scan_csv(directory / "elektriciteitsprod_naar_energiedrager_perioden.csv", separator=";")
dragers = pl.scan_csv(directory / "elektriciteitsprod_naar_energiedrager_dragers.csv", separator=";")
centraal = pl.scan_csv(directory / "elektriciteitsprod_naar_energiedrager_centdecent.csv", separator=";")
groepen = pl.scan_csv(directory / "elektriciteitsprod_naar_energiedrager_categorygroups.csv", separator=";")

merged_dragers = dragers.join(
    groepen.select([
        pl.col("ID"), 
        pl.col("Title").alias("Drager_categorie")
    ]),
    left_on="CategoryGroupID",
    right_on="ID",
    how="left",
)


prod_df = (
    dataset
    .join(
        perioden.select([
            pl.col("Key"), 
            pl.col("Title").alias("Jaar")
        ]),
        left_on="Perioden",
        right_on="Key",
        how="left",
    )
    .join(
        merged_dragers.select([
            pl.col("Key"),
            pl.col("Title").alias("drager"),
            pl.col("Drager_categorie"),
        ]),
        left_on="Energiedragers",
        right_on="Key",
        how="left",
    )
    .join(
        centraal.select([
            pl.col("Key"), 
            pl.col("Title").alias("Centraal")
        ]),
        left_on="CentraleDecentraleProductie",
        right_on="Key",
        how="left",
    )
    .select(
        pl.col("ID"),
        pl.col("Jaar").cast(pl.UInt16),
        pl.col("drager"),
        pl.col("Drager_categorie"),
        pl.col("Centraal"),
        pl.col("TotaalElektriciteitEnWarmteTJ_1").alias("Totaal_TJ"),
        pl.col("ElektriciteitGWh_2").alias("Elektriciteit_GWh"),
        pl.col("ElektriciteitTJ_3").alias("Elektriciteit_TJ"),
        pl.col("Elektriciteit_4").alias("Procent_van_jaar"),
        pl.col("WarmteTJ_5").alias("Warmte_TJ"),
        pl.col("InzetTJ_6").alias("Inzet_TJ"),
        pl.col("ElektrischVermogenMW_7").alias("ElektrischVermogen_MW"),
    )
    .collect()
)
print("sample(n=10, shuffle=False)")
print("-----------------")
print(prod_df.sample(n=10, shuffle=False))

sample(n=10, shuffle=False)
-----------------
shape: (10, 12)
┌─────┬──────┬──────────────────────────────┬────────────────────────┬────────────┬───────────┬───────────────────┬──────────────────┬──────────────────┬───────────┬──────────┬───────────────────────┐
│ ID  ┆ Jaar ┆ drager                       ┆ Drager_categorie       ┆ Centraal   ┆ Totaal_TJ ┆ Elektriciteit_GWh ┆ Elektriciteit_TJ ┆ Procent_van_jaar ┆ Warmte_TJ ┆ Inzet_TJ ┆ ElektrischVermogen_MW │
│ --- ┆ ---  ┆ ---                          ┆ ---                    ┆ ---        ┆ ---       ┆ ---               ┆ ---              ┆ ---              ┆ ---       ┆ ---      ┆ ---                   │
│ u16 ┆ u16  ┆ str                          ┆ str                    ┆ str        ┆ u32       ┆ u32               ┆ u32              ┆ f64              ┆ u32       ┆ u32      ┆ u32                   │
╞═════╪══════╪══════════════════════════════╪════════════════════════╪════════════╪═══════════╪═══════════════════╪══════════════════╪

In [9]:
dataset = pl.scan_csv(
    directory / "emissies_naar_lucht_dataset.csv",
    separator=";",
)
perioden = pl.scan_csv(directory / "emissies_naar_lucht_perioden.csv", separator=";")
bronnen = pl.scan_csv(directory / "emissies_naar_lucht_bronnen.csv", separator=";")
stoffen = pl.scan_csv(directory / "emissies_naar_lucht_stoffen.csv", separator=";")

emi_df = (
    dataset
    .join(
    perioden.select([
        pl.col("Key"), 
        pl.col("Title").alias("Jaar")
    ]),
    left_on="Perioden",
    right_on="Key",
    how="left",
    )
    .join(
    bronnen.select([
        pl.col("Key"), 
        pl.col("Title").alias("Bron")
    ]),
    left_on="Emissiebronnen",
    right_on="Key",
    how="left",
    )
    .join(
    stoffen.select([
        pl.col("Key"), 
        pl.col("Title").alias("Stof")
    ]),
    left_on="EmissiesNaarLucht",
    right_on="Key",
    how="left",
    )
    .select(
        pl.col("ID"),
        pl.col("Jaar"),
        pl.col("Bron"),
        pl.col("Stof"),
        pl.col("EmissiesNaarLucht_1").alias("Uitstoot_mln_kg"),
    )
    .collect()
)

print("sample(n=10, shuffle=False)")
print("-----------------")
print(emi_df.sample(n=10, shuffle=False))

sample(n=10, shuffle=False)
-----------------
shape: (10, 5)
┌──────┬──────┬─────────────────────────────────┬──────────────────────┬─────────────────┐
│ ID   ┆ Jaar ┆ Bron                            ┆ Stof                 ┆ Uitstoot_mln_kg │
│ ---  ┆ ---  ┆ ---                             ┆ ---                  ┆ ---             │
│ i64  ┆ i64  ┆ str                             ┆ str                  ┆ f64             │
╞══════╪══════╪═════════════════════════════════╪══════════════════════╪═════════════════╡
│ 257  ┆ 2003 ┆ Totaal Stationaire en mobiele … ┆ Koolmonoxide (CO)    ┆ 743.5           │
│ 1427 ┆ 2025 ┆ 35 Energiebedrijven             ┆ Kooldioxide (CO2)    ┆ 31600.0         │
│ 1538 ┆ 2024 ┆ 35 Energiebedrijven             ┆ Stikstofoxiden (NOx) ┆ 7.2             │
│ 1609 ┆ 2011 ┆ 35 Energiebedrijven             ┆ PM10 (fijn stof)     ┆ 0.1             │
│ 2188 ┆ 2002 ┆ 10-12 Voedings-, genotmiddelen… ┆ PM2.5 (fijn stof)    ┆ 0.4             │
│ 2258 ┆ 2016 ┆ 13-15 Textiel

In [10]:
pivoted_df = emi_df.pivot(
    "Stof",
    index=["Jaar", "Bron"],
    values="Uitstoot_mln_kg",
)

print("sample(n=10, shuffle=False)")
print("-----------------")
print(pivoted_df.sample(n=10, shuffle=False))

sample(n=10, shuffle=False)
-----------------
shape: (10, 8)
┌──────┬─────────────────────────────────┬───────────────────┬──────────────────────┬─────────────────────┬──────────────────┬───────────────────┬───────────────────┐
│ Jaar ┆ Bron                            ┆ Kooldioxide (CO2) ┆ Stikstofoxiden (NOx) ┆ Zwaveldioxide (SO2) ┆ PM10 (fijn stof) ┆ PM2.5 (fijn stof) ┆ Koolmonoxide (CO) │
│ ---  ┆ ---                             ┆ ---               ┆ ---                  ┆ ---                 ┆ ---              ┆ ---               ┆ ---               │
│ i64  ┆ str                             ┆ f64               ┆ f64                  ┆ f64                 ┆ f64              ┆ f64               ┆ f64               │
╞══════╪═════════════════════════════════╪═══════════════════╪══════════════════════╪═════════════════════╪══════════════════╪═══════════════════╪═══════════════════╡
│ 2016 ┆ Totaal Stationaire en mobiele … ┆ 184500.0          ┆ 355.4                ┆ 33.8              

In [11]:
with pl.Config(tbl_rows=-1):
    print(pivoted_df.filter(pl.col("Jaar") >= 2024))

shape: (48, 8)
┌──────┬─────────────────────────────────┬───────────────────┬──────────────────────┬─────────────────────┬──────────────────┬───────────────────┬───────────────────┐
│ Jaar ┆ Bron                            ┆ Kooldioxide (CO2) ┆ Stikstofoxiden (NOx) ┆ Zwaveldioxide (SO2) ┆ PM10 (fijn stof) ┆ PM2.5 (fijn stof) ┆ Koolmonoxide (CO) │
│ ---  ┆ ---                             ┆ ---               ┆ ---                  ┆ ---                 ┆ ---              ┆ ---               ┆ ---               │
│ i64  ┆ str                             ┆ f64               ┆ f64                  ┆ f64                 ┆ f64              ┆ f64               ┆ f64               │
╞══════╪═════════════════════════════════╪═══════════════════╪══════════════════════╪═════════════════════╪══════════════════╪═══════════════════╪═══════════════════╡
│ 2024 ┆ Totaal Stationaire en mobiele … ┆ 140400.0          ┆ 268.7                ┆ 18.6                ┆ 28.5             ┆ 16.0              ┆ 406